# Calculate textual features in Aus, Canada, UK data

### Load libraries and import data

In [1]:
# load libraries
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from dateutil.relativedelta import relativedelta
from datetime import date, timedelta
import statistics
import re
from collections import Counter
import re
import contractions
import string
import textstat
from readability import Readability
from scipy.stats import median_test
import scipy.stats as stats

In [2]:
# Read in dataset

# Australia
with open('../data/aus/clean/aus_deidentified.pkl', 'rb') as f:
    aus = pickle.load(f, encoding="utf-8")
    
# Canada
with open('../data/canada/clean/can_deidentified.pkl', 'rb') as f:
    can = pickle.load(f, encoding="utf-8")
    
# UK
with open('../data/uk/clean/uk_deidentified.pkl', 'rb') as f:
    uk = pickle.load(f, encoding="utf-8")

### Create body text blocks and title text blocks

In [3]:
# create keyword text blocks - not using 'title' fields for Australia, and UK

aus_text = {}
for k in aus.keys():
    texts = []
    if aus[k]['reason'] != None:
        texts.append(aus[k]['reason']) #.lower())
    if aus[k]['request'] != None:
        texts.append(aus[k]['request']) #.lower())
    aus_text[k] = ' '.join(texts)
    
can_text = {}
for k in can.keys():
    texts = []
    if can[k]['Grievances'] != None:
        texts.append(can[k]['Grievances'])
    if can[k]['Prayer'] != None:
        texts.append(can[k]['Prayer'])
    can_text[k] = ' '.join(texts)
    
uk_text = {}
for k in uk.keys():
    texts = []
    if uk[k]['background'] != None:
        texts.append(uk[k]['background'])
    if uk[k]['additional_details'] != None:
        texts.append(uk[k]['additional_details'])
    uk_text[k] = ' '.join(texts)
    
    
aus_titles = {}
for k in aus.keys():
    if 'A petition from' not in aus[k]['title'] and aus[k]['title']!= None:
        aus_titles[k] = aus[k]['title'] #.lower())
    else:
        aus_titles[k] = None
               
can_titles = None

uk_titles = {}
for k in uk.keys():
    uk_titles[k] = uk[k]['action']

### Prepare valence and arousal data from Warriner et al. (2013)

In [4]:
# Valence and Arousal word scores from paper
f = pd.read_csv('../data/BRM-emot-submit.csv')

val_arous_dict = f[['Word', 'V.Mean.Sum', 'A.Mean.Sum']].to_dict()
val_arous_dict_fin = {}
for k in val_arous_dict['Word'].keys():
    val_arous_dict_fin[val_arous_dict['Word'][k]] = {'arousal': val_arous_dict['A.Mean.Sum'][k],
                                                     'valence': val_arous_dict['V.Mean.Sum'][k]}


In [5]:
pd.DataFrame(val_arous_dict)

,Word,V.Mean.Sum,A.Mean.Sum
0,aardvark,6.26,2.41
1,abalone,5.30,2.65
2,abandon,2.84,3.73
3,abandonment,2.63,4.95
4,abbey,5.85,2.20
...,...,...,...
13910,zone,4.75,3.78
13911,zoning,4.65,3.77
13912,zoo,7.00,5.63
13913,zoom,5.86,5.68


### Code for processing text and calculating textual features

In [6]:
def process_text(text):
    # get rid of whitespace
    split = text.split()

    # expand contractions
    text = ' '.join([contractions.fix(word) for word in split])
    
    # remove urls
    text = re.sub('https?://\S+|http?://\S+|www\S+', '',text)
    
    # remove punctuation    
    translator = str.maketrans('', '', string.punctuation)

    text = text.translate(translator)
    text = text.replace("’", '')
    
    # remove white space and make lowercase
    split = [x.strip() for x in text.split() if len(x) >= 1]

    # remove numbers (though NOT alpha-numeric words like 5G)
    split = [t for t in split if not all(i.isnumeric() for i in t)]

    split_lower = [x.lower() for x in split]
    
    return split_lower, split

In [7]:
def herdan_c(text):
    if len(text) == 1:
        c = None
    else:
        c = np.log(len(set(text)))/np.log(len(text))
    
    return c

In [8]:
import spacy
nlp = spacy.load("en_core_web_sm")

def create_lems(text): # list of words
    text = nlp(' '.join(text))
    
    lemmatized = []
    
    for token in text:
        if len(token) > 0:
            lemmatized.append(token.lemma_,)
    
    return lemmatized

def valence_arousal_dominance(val_dict, text):
    # need to remove stop words and lemmatize perhaps
    
    text = create_lems(text)

    if len(text) > 0:
        counter = 0
        v = 0
        a = 0
        for x in text:
            if x in val_dict:
                a += val_dict[x]['arousal']
                v += val_dict[x]['valence']
                counter += 1
    
        if counter == 0:
            # return v, a, v/len(text), a/len(text)
            return None, None, None, None
        else:
            return v/counter, a/counter, v/len(text), a/len(text)
    else:
        return None, None, None, None
            

In [9]:
## Function to carry

def feature_pipeline(pets, titles, val_dict, full_epets, gen_field, sig_field):

    
    feature_dict = {'ID': [],
                    'Gender': [],
                    'Signatures': [],
                   'Length': [], 
                   'Length_Unique': [],
                    'Repetition': [], 
                   'Herdan_C': [],
                    'Valence_Avg':[],
                    'Arousal_Avg': [],
                    'Title_Length': []}
    
    for k in pets.keys():      
        feature_dict['ID'].append(k)
        if gen_field != None:
            feature_dict['Gender'].append(full_epets[k][gen_field])
        else:
            feature_dict['Gender'].append(None)
            
        feature_dict['Signatures'].append(full_epets[k][sig_field])
                
        text = pets[k]
            
        l,l_caps_pres = process_text(pets[k]) # fully processed text
        
        length = len(l) # length of text
        length_unique = len(set(l)) # number of unique words in text
        if length_unique > 0:
            repetition = length/length_unique # ratio
        else:
            repetition = None    
        feature_dict['Length'].append(length)
        feature_dict['Length_Unique'].append(length_unique)
        feature_dict['Repetition'].append(repetition)
        

        # text complexity
        if length > 0:
            c = herdan_c(l)
        else:
            c = None
        feature_dict['Herdan_C'].append(c)
        
        # Petition Title Features - only for countries with titles
        if titles != None:
            if titles[k] != None:
                title_l, title_l_caps = process_text(titles[k])
                title_length = len(title_l)
                feature_dict['Title_Length'].append(title_length)
            else:
                feature_dict['Title_Length'].append(None)
        else:
            feature_dict['Title_Length'].append(None)
                      
               
        # Readibility - Valence/Arousal - Request - Urgency - Extremity
        textstat.set_lang('en')
        # valence and arousal 
        v, a, v_fulldenom, a_fulldenom = valence_arousal_dominance(val_dict, l)
        feature_dict['Valence_Avg'].append(v)
        feature_dict['Arousal_Avg'].append(a) 

        # print(k)
        
    return feature_dict
        

### For each individual feature in each country, fit a linear regression model

In [10]:
def lin_reg_process(df_in, fields, filt_flag = False, round_flag = 5):
    corr_dict = {'fields': fields,
                'reg_slope': [],
                 'reg_slope_se': [],
                'reg_int': [],
                'reg_pval': [],
                 'slope_conf_int_str':[],
                 'slope_conf_int_str2':[],
                'r': [],
                'r2': [],
                'adj_r2': []}
    
    df = df_in.copy()
    df['Signatures_Log'] = [np.log(x) for x in df['Signatures']]
    
    
    print(f"DF length: {len(df)}")
    
    # add filter, if filter flag is True, to check and remove N/A rows across all features
    if filt_flag == True:
        df = df.dropna(subset=fields)
        print(f"DF length after filtering for non-NA in feature fields: {len(df)}")

    for field in fields:
        # drop NA values
        df_filt = df.dropna(subset=[field, 'Signatures_Log'])
        
        # add in linear regression
        slope, intercept, r, p, se = stats.linregress(df_filt[field], df_filt['Signatures_Log'], nan_policy='omit')

        
        r2 = r**2
        n = len(df_filt)
        print(f"Field {field}: {n}")
        k = 1
        adj_r2 = 1 - (1-r2)*((n-1)/(n - k - 1))

        
        from scipy.stats import t
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, n-2) 
        conf_int_string = fr"{slope:.5f} $\pm$ {ts*se:.5f} ({se:.5f})"
        
        conf_int_string2 = fr"{slope:.3f} [{slope - ts*se:.3f}, {slope + ts*se:.3f}] ({se:.3f})"

        corr_dict['reg_slope'].append(round(slope, round_flag))
        corr_dict['reg_slope_se'].append(round(se, round_flag))
        corr_dict['reg_int'].append(round(intercept, round_flag))
        corr_dict['reg_pval'].append(round(p, round_flag))
        corr_dict['slope_conf_int_str'].append(conf_int_string)
        corr_dict['slope_conf_int_str2'].append(conf_int_string2)

        corr_dict['r'].append(round(r, round_flag))
        corr_dict['r2'].append(round(r2, round_flag))
        corr_dict['adj_r2'].append(round(adj_r2, round_flag))
        
        
    return corr_dict


### Full pipeline: process text, calculate features, fit linear regression model on each individual feature, output LaTeX table of results, calculate gender-based feature tests for Australia and Canada

In [11]:
def full_text_pipeline(text, titles, val_arous_dict_fin, epets, fields, gen_field, sig_field, filt_flag = False, round_flag = 5):
    from scipy.stats import median_test
    import scipy.stats as stats
    feature_dict = feature_pipeline(text, titles, val_arous_dict_fin, epets, gen_field, sig_field)
    feature_dict_df = pd.DataFrame(feature_dict)

    # this line was to check N/As in Aus
    # print(f"N/A in Title Length: {len(feature_dict_df[feature_dict_df['Title_Length'].isna()])}")
    
    if gen_field != None:
        l1 = feature_dict_df[feature_dict_df['Gender'] == 'female']['Length_Unique']
        l2 = feature_dict_df[feature_dict_df['Gender'] == 'male']['Length_Unique']
        
        print(f"female length-unique N: {len(l1)}")
        # print(l1.isna().sum())
        print(f"male length-unique N: {len(l2)}")
        # print(l2.isna().sum())
        
        print(median_test(l1, l2))
        statistic, p_value = stats.mannwhitneyu(l1, l2, alternative='two-sided')
        print(f"Test Statistic: {statistic}")
        print(f"P-value: {p_value}")
    
    corr_dict = lin_reg_process(feature_dict_df, fields, filt_flag, round_flag)
    
    corr_df = pd.DataFrame(corr_dict).sort_values(by = 'adj_r2', ascending = False)
    corr_df['reg_slope_tstat'] = corr_df['reg_slope']/corr_df['reg_slope_se']
    
    print(corr_df[['fields', 'adj_r2', 'slope_conf_int_str2','reg_pval']].to_latex())

In [12]:
fields = ['Length', 'Length_Unique', 'Repetition', 'Herdan_C', 'Valence_Avg', 'Arousal_Avg', 'Title_Length']
full_text_pipeline(aus_text, aus_titles, val_arous_dict_fin, aus, fields, 'gender', 'signatures', False)

# fields = ['Length', 'Length_Unique', 'Repetition', 'Herdan_C', 'Valence_Avg', 'Arousal_Avg']
# full_text_pipeline(aus_text, aus_titles, val_arous_dict_fin, aus, fields, 'gender', 'signatures', True)

female length-unique N: 1514
male length-unique N: 2366
MedianTestResult(statistic=0.57540447158004, pvalue=0.44811928008967883, median=101.0, table=array([[ 762, 1160],
       [ 752, 1206]]))
Test Statistic: 1872090.5
P-value: 0.017281145909992135
DF length: 3906
Field Length: 3906
Field Length_Unique: 3906
Field Repetition: 3906
Field Herdan_C: 3906
Field Valence_Avg: 3906
Field Arousal_Avg: 3906
Field Title_Length: 3346
\begin{tabular}{llrlr}
\toprule
 & fields & adj_r2 & slope_conf_int_str2 & reg_pval \\
\midrule
1 & Length_Unique & 0.049880 & 0.014 [0.012, 0.016] (0.001) & 0.000000 \\
0 & Length & 0.048160 & 0.008 [0.006, 0.009] (0.001) & 0.000000 \\
6 & Title_Length & 0.018080 & 0.133 [0.100, 0.165] (0.017) & 0.000000 \\
2 & Repetition & 0.013670 & 1.261 [0.928, 1.593] (0.170) & 0.000000 \\
3 & Herdan_C & 0.004950 & -7.666 [-10.993, -4.339] (1.697) & 0.000010 \\
5 & Arousal_Avg & 0.003850 & 0.897 [0.458, 1.336] (0.224) & 0.000060 \\
4 & Valence_Avg & 0.003050 & -0.513 [-0.793, -0

In [24]:
fields = ['Length', 'Length_Unique', 'Repetition', 'Herdan_C', 'Valence_Avg', 'Arousal_Avg']
full_text_pipeline(can_text, can_titles, val_arous_dict_fin, can, fields, 'gender', 'Signatures', True)

female length-unique N: 932
male length-unique N: 1061
MedianTestResult(statistic=13.389413280093667, pvalue=0.0002530485339611488, median=113.0, table=array([[500, 481],
       [432, 580]]))
Test Statistic: 544345.0
P-value: 9.841781604792211e-05
DF length: 2034
DF length after filtering for non-NA in feature fields: 2034
Field Length: 2034
Field Length_Unique: 2034
Field Repetition: 2034
Field Herdan_C: 2034
Field Valence_Avg: 2034
Field Arousal_Avg: 2034
\begin{tabular}{llrlr}
\toprule
 & fields & adj_r2 & slope_conf_int_str2 & reg_pval \\
\midrule
1 & Length_Unique & 0.028160 & 0.008 [0.006, 0.011] (0.001) & 0.000000 \\
0 & Length & 0.025190 & 0.004 [0.003, 0.005] (0.001) & 0.000000 \\
4 & Valence_Avg & 0.001420 & -0.227 [-0.453, -0.001] (0.115) & 0.048900 \\
5 & Arousal_Avg & 0.000870 & 0.315 [-0.056, 0.685] (0.189) & 0.096200 \\
2 & Repetition & 0.000800 & 0.299 [-0.063, 0.661] (0.185) & 0.105080 \\
3 & Herdan_C & -0.000370 & 0.896 [-2.664, 4.455] (1.815) & 0.621690 \\
\bottomrul

In [25]:
fields = ['Length', 'Length_Unique', 'Repetition', 'Herdan_C', 'Valence_Avg', 'Arousal_Avg', 'Title_Length']
full_text_pipeline(uk_text, uk_titles, val_arous_dict_fin, uk, fields, None, 'signatures', True)

DF length: 52154
DF length after filtering for non-NA in feature fields: 52148
Field Length: 52148
Field Length_Unique: 52148
Field Repetition: 52148
Field Herdan_C: 52148
Field Valence_Avg: 52148
Field Arousal_Avg: 52148
Field Title_Length: 52148
\begin{tabular}{llrlr}
\toprule
 & fields & adj_r2 & slope_conf_int_str2 & reg_pval \\
\midrule
1 & Length_Unique & 0.028670 & 0.018 [0.017, 0.018] (0.000) & 0.000000 \\
0 & Length & 0.027850 & 0.010 [0.010, 0.011] (0.000) & 0.000000 \\
2 & Repetition & 0.017270 & 1.655 [1.547, 1.762] (0.055) & 0.000000 \\
6 & Title_Length & 0.013820 & 0.075 [0.070, 0.081] (0.003) & 0.000000 \\
3 & Herdan_C & 0.013540 & -10.838 [-11.631, -10.044] (0.405) & 0.000000 \\
5 & Arousal_Avg & 0.001130 & 0.412 [0.308, 0.517] (0.053) & 0.000000 \\
4 & Valence_Avg & 0.000870 & -0.231 [-0.298, -0.165] (0.034) & 0.000000 \\
\bottomrule
\end{tabular}

